In [1]:
import torch
import torch.nn as nn
import torch.quantization as tq
from transformers import AutoModelForCausalLM, AutoTokenizer

### Load a smaller LLM (for demo we use DistilGPT2)

In [5]:
# Step 1: Load a smaller LLM (for demo we use DistilGPT2)
model_name = "distilgpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step 2: Define QAT config
# qat_config = tq.QConfig(
#     activation=tq.FakeQuantize.with_args(observer=tq.MovingAverageMinMaxObserver, quant_min=0, quant_max=255, dtype=torch.quint8, qscheme=torch.per_tensor_affine),
#     weight=tq.FakeQuantize.with_args(observer=tq.MinMaxObserver, quant_min=-128, quant_max=127, dtype=torch.qint8, qscheme=torch.per_tensor_symmetric)
# )

qat_config = tq.QConfig(
    activation=tq.FakeQuantize.with_args(
        observer=tq.MovingAverageMinMaxObserver,
        quant_min=0, quant_max=255,
        dtype=torch.quint8, qscheme=torch.per_tensor_affine
    ),
    weight=tq.FakeQuantize.with_args(
        observer=tq.MinMaxObserver,
        quant_min=-128, quant_max=127,
        dtype=torch.qint8, qscheme=torch.per_tensor_symmetric
    )
)


# Explicitly ignore embedding layers
for name, module in model.named_modules():
    if isinstance(module, nn.Embedding):
        module.qconfig = None  # no quantization for embeddings


# Step 3: Attach quant/dequant stubs
model.qconfig = qat_config

model.train()
tq.prepare_qat(model, inplace=True)   # Model now has fake quant ops

# Step 4: Fine-tune the quantization-aware model
inputs = tokenizer("Quantization Aware Training on LLMs!", return_tensors="pt")

labels = inputs["input_ids"]

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# model.train()

for step in range(50):   # tiny fine-tuning loop
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if step % 10 == 0:
        print(f"Step {step} | Loss: {loss.item()}")

# Step 5: Convert to fully quantized
qat_model = tq.convert(model.eval(), inplace=False)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

/tmp/ipykernel_1126454/1058848321.py:36: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  tq.prepare_qat(model, inplace=True)   # Model now has fake quant ops
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step 0 | Loss: 9.036910057067871
Step 10 | Loss: 2.3155338764190674
Step 20 | Loss: 0.7709331512451172
Step 30 | Loss: 0.17784838378429413
Step 40 | Loss: 1.3359317779541016


/tmp/ipykernel_1126454/1058848321.py:57: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  qat_model = tq.convert(model.eval(), inplace=False)
/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/torch/ao/nn/quantized/modules/utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions

RuntimeError: unknown architecure